# Source-Aware OSINT Agent with Pydantic AI

The presentation shows a more production-like system: case folders, `AGENTS.md`, an investigation playbook, separate skills, logs, and reports. This notebook keeps the same idea, but compresses it into one Colab-friendly flow:

**case input → source-aware agent → registry/database tools + Pydantic provider-native web search → structured report**


## How this maps to the presentation

| In the presentation | In this notebook |
|---|---|
| `AGENTS.md` | The agent instructions cell |
| `INVESTIGATION_PLAYBOOK.md` | Tool docstrings + workflow rules |
| `skills/*/SKILL.md` | Custom Python tools such as YC World and Aleph lookup |
| Orchestrator skill | The Pydantic AI `Agent` deciding what to call |
| Case folder / `CASE.md` | The company + question prompt |
| Final investigation report | The `ResearchReport` Pydantic output model |

The biggest simplification: we do **not** write a folder-based state machine here. We let Pydantic AI handle provider-native web search and tool calling, then we force the answer into a structured report.


In [ ]:
# Setup cell
%pip install -q requests==2.32.5 "pydantic-ai-slim[openai]==0.8.1"


In [ ]:
# Configure keys and shared settings
# Paste keys into the empty strings below only in a private local copy.

import os
import re
from getpass import getpass
from pathlib import Path
from typing import Literal

import requests
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.builtin_tools import WebSearchTool
from pydantic_ai.models.openai import OpenAIResponsesModel
from pydantic_ai.settings import ModelSettings

OPENAI_API_KEY = ""
YC_WORLD_API_KEY = ""
ALEPH_API_KEY = ""
OPENAI_MODEL = "gpt-4.1-mini"
YC_WORLD_BASE_URL = "https://api.youcontrol.world"
ALEPH_BASE_URL = "https://aleph.occrp.org"
OUTPUT_DIR = Path("outputs")

for key, value in [
    ("OPENAI_API_KEY", OPENAI_API_KEY),
    ("YC_WORLD_API_KEY", YC_WORLD_API_KEY),
    ("ALEPH_API_KEY", ALEPH_API_KEY),
]:
    if value:
        os.environ[key] = value

for key, label in [
    ("OPENAI_API_KEY", "OpenAI API key"),
    ("YC_WORLD_API_KEY", "YC World API key (Enter to skip)"),
    ("ALEPH_API_KEY", "Aleph API key (Enter to skip; public search may still work)"),
]:
    if not os.getenv(key):
        value = getpass(f"{label}: ")
        if value:
            os.environ[key] = value

OPENAI_MODEL = os.getenv("OPENAI_MODEL", OPENAI_MODEL)
YC_WORLD_API_KEY = os.getenv("YC_WORLD_API_KEY", YC_WORLD_API_KEY)
ALEPH_API_KEY = os.getenv("ALEPH_API_KEY", ALEPH_API_KEY) or os.getenv("ALEPHCLIENT_API_KEY", "")
YC_WORLD_BASE_URL = os.getenv("YC_WORLD_BASE_URL", YC_WORLD_BASE_URL).rstrip("/")
ALEPH_BASE_URL = os.getenv("ALEPH_BASE_URL", ALEPH_BASE_URL).rstrip("/")

print(f"Model: OpenAI Responses API / {OPENAI_MODEL}")
print(f"YC World: {'enabled' if YC_WORLD_API_KEY else 'disabled'}")
print(f"Aleph: {'key provided' if ALEPH_API_KEY else 'public/no-key mode'}")


## Report Shape

This cell is like the **report template**.

Pydantic forces the model to separate facts, claims, leads, hypotheses, and uncertainty. This is the main reason Pydantic AI is useful for teaching OSINT: it discourages one big blurry answer.


In [ ]:
# Structured output = the contract the agent must satisfy.
# The field names are intentionally plain because students will read them.

class ResearchReport(BaseModel):
    short_answer: str = Field(description="A cautious direct answer to the research question.")
    registry_facts: list[str] = Field(default_factory=list, description="Facts supported by registry or database records.")
    public_claims: list[str] = Field(default_factory=list, description="Claims from public sources such as articles, reports, websites, or PDFs.")
    leads_to_review: list[str] = Field(default_factory=list, description="Promising but unconfirmed leads that need manual review.")
    risk_hypotheses: list[str] = Field(default_factory=list, description="Reasoned hypotheses, clearly not conclusions.")
    uncertainty: list[str] = Field(default_factory=list, description="What remains unclear or weakly supported.")
    next_steps: list[str] = Field(default_factory=list, description="Specific manual checks a reporter should do next.")
    source_ledger: list[str] = Field(default_factory=list, description="URLs, source names, record IDs, or database names used.")


## Custom Tools

This cell is like the **skills folder**.

The notebook now has only special-source tools here. General web search is exposed through Pydantic AI's provider-native `WebSearchTool`. This is different from the local DuckDuckGo common tool; use `pydantic_ai.common_tools.duckduckgo` if you want local DuckDuckGo search instead.


In [ ]:
# Helper used by the registry tools.

def first(value):
    if isinstance(value, list):
        return str(value[0]) if value else ""
    return str(value or "")


def yc_world_lookup(query: str, schema: Literal["Company", "Person"] = "Company", country: str = "") -> str:
    """
    Search YC World registry records. Use schema='Company' or schema='Person'.

    Use for company registry facts, directors, owners, addresses,
    registration numbers, and related entities.

    Do not use for media claims, sanctions context, or broad background research.

    Warning: registry matches are leads until manually reviewed. Include source URLs,
    YC World IDs, and other source identifiers in the report ledger.
    """
    if not YC_WORLD_API_KEY:
        return "YC World is disabled because YC_WORLD_API_KEY is not set."

    params = {"SearchString": query, "SchemaName": schema, "PageSize": 5, "Offset": 0}
    if country:
        params["Countries"] = country.lower()

    response = requests.get(
        f"{YC_WORLD_BASE_URL}/GetEntities",
        params=params,
        headers={"Accept": "application/json", "x-api-key": YC_WORLD_API_KEY},
        timeout=45,
    )
    response.raise_for_status()
    payload = response.json().get("result", response.json())

    lines = []
    for entity in payload.get("entities", [])[:5]:
        item = (entity.get("items") or [{}])[0]
        props = item.get("properties", {}) or {}
        external_id = entity.get("externalId", "")

        lines.append(f"Entity: {first(props.get('name')) or entity.get('caption', '?')}")
        lines.append(f"  Country: {first(props.get('country'))}")
        lines.append(f"  Status: {first(props.get('status'))}")
        lines.append(f"  Address: {first(props.get('address'))}")
        lines.append(f"  Source URL: {first(props.get('sourceUrl'))}")
        lines.append(f"  YC World ID: {external_id}")
        lines.extend(yc_world_relations(external_id))
        lines.append("")

    return "\n".join(lines) or "No YC World results found."


def yc_world_relations(external_id: str) -> list[str]:
    """Fetch a small set of related entities for a YC World entity."""
    if not external_id:
        return []

    response = requests.get(
        f"{YC_WORLD_BASE_URL}/Entity/{external_id}/Relations",
        headers={"Accept": "application/json", "x-api-key": YC_WORLD_API_KEY},
        timeout=45,
    )
    if not response.ok:
        return ["  Relations: unavailable"]

    data = response.json().get("result", response.json())
    relations = data.get("relations", []) if isinstance(data, dict) else []
    lines = []
    for relation in relations[:5]:
        target = relation.get("target", {}) or relation.get("entity", {}) or {}
        caption = target.get("caption") or target.get("name") or "unknown"
        role = relation.get("type") or relation.get("role") or "related"
        lines.append(f"  Relation: {role} -> {caption}")
    return lines


In [ ]:
# Aleph = another special-source skill.
# The output is text because it is easier for the agent to read during tool calls.

def aleph_lookup(query: str, schema: Literal["Company", "Person"] = "Company") -> str:
    """
    Search Aleph investigative datasets. Use schema='Company' or schema='Person'.

    Use for investigative datasets, sanctions-style records, leaks,
    PEP-style links, cross-border leads, and related entities.

    Do not use for basic public background that should come from public web sources.

    Warning: Aleph hits are leads unless the underlying dataset is inspected.
    Include Aleph URLs, entity IDs, and dataset names in the report ledger.
    """
    headers = {"Accept": "application/json"}
    if ALEPH_API_KEY:
        headers["Authorization"] = f"ApiKey {ALEPH_API_KEY}"

    response = requests.get(
        f"{ALEPH_BASE_URL}/api/2/entities",
        params={"q": query, "filter:schema": schema, "limit": 5},
        headers=headers,
        timeout=45,
    )
    response.raise_for_status()
    rows = response.json().get("results", [])[:5]

    lines = []
    for item in rows:
        links = item.get("links", {}) or {}
        lines.append(f"Entity: {item.get('caption', '?')} ({item.get('schema', '')})")
        lines.append(f"  URL: {links.get('ui', '')}")
        lines.append(f"  Aleph ID: {item.get('id', '')}")
        lines.append("")

    return "\n".join(lines) or "No Aleph results found."


## The Agent

This cell is like `AGENTS.md` plus a simple orchestrator.


In [ ]:
# Concise instructions usually work better than long narrative prompts.
# Tool docstrings carry source-specific guidance.

AGENT_INSTRUCTIONS = """
You are a cautious OSINT research assistant.

Goal:
Answer the user's company-research question using source-led evidence.

Workflow:
1. Use the custom tools according to their docstrings.
2. Use the provider-native WebSearchTool for public reports, media, company websites, PDFs,
   NGO reports, and official pages.
3. Do not treat search snippets alone as confirmed evidence; prefer sources with URLs or readable source context.
4. Use at least two source types when possible.
5. Follow only the strongest related-person or related-company leads.

Evidence rules:
- Treat matches as leads unless directly confirmed by a source.
- Separate facts, claims, leads, hypotheses, uncertainty, and next steps.
- Include URLs, source IDs, record IDs, or database names in the source ledger.
- Say clearly when evidence is weak, missing, contradictory, or ambiguous.
"""

agent = Agent(
    OpenAIResponsesModel(OPENAI_MODEL),
    output_type=ResearchReport,
    # Provider-native web search, not local DuckDuckGo.
    builtin_tools=[WebSearchTool()],
    tools=[yc_world_lookup, aleph_lookup],
    instructions=AGENT_INSTRUCTIONS,
    model_settings=ModelSettings(temperature=0),
)


## Run It

This cell is like the **case file**.

Keep the run prompt short. The agent already has the workflow and evidence rules above, so the prompt only needs the target and the question.


In [ ]:
def slug(text: str) -> str:
    return re.sub(r"[^0-9a-zA-Z]+", "_", text.lower()).strip("_") or "company"


def save_report(company: str, question: str, report: ResearchReport) -> Path:
    folder = OUTPUT_DIR / slug(company)
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / "report.md"

    sections = [
        ("Registry Facts", report.registry_facts),
        ("Public Claims", report.public_claims),
        ("Leads To Review", report.leads_to_review),
        ("Risk Hypotheses", report.risk_hypotheses),
        ("Uncertainty", report.uncertainty),
        ("Next Steps", report.next_steps),
        ("Source Ledger", report.source_ledger),
    ]

    lines = [
        f"# OSINT report: {company}",
        "",
        f"Question: {question}",
        "",
        "## Short Answer",
        "",
        report.short_answer,
        "",
    ]

    for title, items in sections:
        if items:
            lines.extend([f"## {title}", "", *[f"- {item}" for item in items], ""])

    path.write_text("\n".join(lines), encoding="utf-8")
    return path


async def run_investigation(company: str, question: str, country: str = "") -> ResearchReport:
    prompt = f"""
Target company: {company}
Country: {country or "unknown"}
Research question: {question}

Produce a cautious source-led report.
"""
    result = await agent.run(prompt)
    report = result.output
    print(report.short_answer)
    print(f"\nSaved: {save_report(company, question, report)}")
    return report


In [ ]:
report = await run_investigation(
    company="CONSTEEL ELECTRONICS SP Z O O",
    country="Poland",
    question="Who owns or controls this company, and are there any risk-relevant public claims or leads?"
)